In [ ]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

assert torch.cuda.is_available(), "Please enable GPU in the runtime settings first."

gpu = torch.cuda.get_device_properties(0)
print("GPU:", gpu.name)
print("Video memory capacity GB:", round(gpu.total_memory / 1024**3, 2))

PyTorch: 2.11.0+cu128
CUDA available: True
GPU: Tesla T4
显存容量 GB: 14.56


In [ ]:
%pip install -q -U transformers accelerate bitsandbytes

In [ ]:
import torch
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
)

MODEL_ID = "Qwen/Qwen3-1.7B"

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

tokenizer.padding_side = "left"

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=quantization_config,
    device_map="auto",
    dtype=torch.float16,
)

model.eval()

print("Model loading successful:", MODEL_ID)
print("Device assign:", model.hf_device_map)

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

模型加载成功: Qwen/Qwen3-1.7B


AttributeError: 'Qwen3ForCausalLM' object has no attribute 'hf_device_map'

In [ ]:
import torch
import bitsandbytes as bnb

devices = sorted({str(p.device) for p in model.parameters()})
quantized_layers = sum(
    isinstance(layer, bnb.nn.Linear4bit)
    for layer in model.modules()
)

print("The device where the parameters are located:", devices)
print("4-bit Quantify the number of layers:", quantized_layers)
print(
    "The current allocated video memory is GB:",
    round(torch.cuda.memory_allocated() / 1024**3, 2)
)

参数所在设备: ['cuda:0']
4-bit 量化层数量: 196
当前已分配显存 GB: 4.47


In [ ]:
from transformers import GenerationConfig

messages = [
    {
        "role": "system",
        "content": "Answer concisely. Output only the final answer."
    },
    {
        "role": "user",
        "content": "What is 15 + 27? Return only the number."
    },
]

prompt = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=False,
)

inputs = tokenizer(
    prompt,
    return_tensors="pt",
    add_special_tokens=False,
).to(model.device)

generation_config = GenerationConfig(
    max_new_tokens=128,
    do_sample=False,
    eos_token_id=model.generation_config.eos_token_id,
    pad_token_id=tokenizer.pad_token_id,
)

with torch.inference_mode():
    outputs = model.generate(
        **inputs,
        generation_config=generation_config,
    )

new_tokens = outputs[0, inputs["input_ids"].shape[1]:]
answer = tokenizer.decode(
    new_tokens,
    skip_special_tokens=True,
).strip()

print("Model asnwers:", answer)

模型回答: 32


In [ ]:
print("Model:", getattr(model.config, "_name_or_path", "Unknow"))
print("Tokenizer:", tokenizer.name_or_path)
print("Practical question:", messages[-1]["content"])

print(
    "The actual text that is input into the model:",
    repr(tokenizer.decode(
        inputs["input_ids"][0],
        skip_special_tokens=False,
    ))
)

print("Original answer:", repr(answer))
print("Is sampling conducted:", generation_config.do_sample)

模型: Qwen/Qwen3-1.7B
Tokenizer: Qwen/Qwen3-1.7B
实际问题: What is 15 + 27? Return only the number.
实际传入模型的文本: '<|im_start|>system\nAnswer concisely. Output only the final answer.<|im_end|>\n<|im_start|>user\nWhat is 15 + 27? Return only the number.<|im_end|>\n<|im_start|>assistant\n<think>\n\n</think>\n\n'
原始答案: '32'
是否采样: False


In [ ]:
import torch
from transformers import AutoModelForCausalLM
from importlib.metadata import version

for package in ["transformers", "accelerate", "bitsandbytes"]:
    print(package, version(package))

answer_4bit = answer
revision = getattr(model.config, "_commit_hash", None)

fp16_model = AutoModelForCausalLM.from_pretrained(
    "Qwen/Qwen3-1.7B",
    revision=revision or "main",
    dtype=torch.float16,
    device_map={"": 0},
)
fp16_model.eval()

fp16_inputs = {
    key: value.to("cuda:0")
    for key, value in inputs.items()
}

with torch.inference_mode():
    fp16_outputs = fp16_model.generate(
        **fp16_inputs,
        generation_config=generation_config,
    )

new_tokens = fp16_outputs[
    0, fp16_inputs["input_ids"].shape[1]:
]

answer_fp16 = tokenizer.decode(
    new_tokens,
    skip_special_tokens=True,
).strip()

print("4-bit Original answer:", repr(answer_4bit))
print("FP16 Compared answer:", repr(answer_fp16))
print("Correct answer: 42")

transformers 5.21.0
accelerate 1.15.0
bitsandbytes 0.50.2


Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

4-bit 原始回答: '32'
FP16 对照回答: '42'
正确参考答案: 42


In [ ]:
from google.colab import files
files.upload()

Saving run_colab_smoke.py to run_colab_smoke.py
Saving smoke_inputs.csv to smoke_inputs.csv


{'run_colab_smoke.py': b'"""Run in the existing Colab session with: %run -i run_colab_smoke.py\n\nUpload this file and smoke_inputs.csv. The FP16 model and tokenizer must already\nexist in the current notebook. No model is downloaded or trained by this script.\n"""\n\nimport csv\nimport gc\nimport hashlib\nimport json\nimport shutil\nimport time\nfrom collections import Counter\nfrom datetime import datetime, timezone\nfrom importlib.metadata import version\nfrom pathlib import Path\n\nimport torch\nfrom transformers import GenerationConfig\n\n\nif "fp16_model" not in globals() or "tokenizer" not in globals():\n    raise RuntimeError("Run in the existing notebook where fp16_model and tokenizer were loaded.")\n\n# Use the already loaded FP16 candidate for all ten examples.\nmodel = fp16_model\nmodel.eval()\ngc.collect()\ntorch.cuda.empty_cache()\nif not torch.cuda.is_available():\n    raise RuntimeError("A CUDA GPU is required for this diagnostic.")\nif any("Linear4bit" in type(layer)._

In [ ]:
%run -i run_colab_smoke.py

[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


Environment: {
  "model_id": "Qwen/Qwen3-1.7B",
  "model_revision": null,
  "tokenizer_id": "Qwen/Qwen3-1.7B",
  "precision": "float16",
  "quantized": false,
  "gpu": "Tesla T4",
  "gpu_capacity_gib": 14.56219482421875,
  "torch": "2.11.0+cu128",
  "cuda": "12.8",
  "packages": {
    "transformers": "5.21.0",
    "accelerate": "1.15.0",
    "bitsandbytes": "0.50.2"
  },
  "parameter_devices": [
    "cuda:0"
  ],
  "model_parameter_and_buffer_gib": 3.2048211097717285,
  "seed": 5494,
  "enable_thinking": false,
  "do_sample": false,
  "batch_size": 1,
  "max_new_tokens": 128,
  "total_token_budget": 4096,
  "padding_side": "left",
  "truncation": "none; fail explicitly if budget exceeded",
  "input_sha256": "c1cd1ffb09c3abec01c19503de8a99ca9c81b5078248a11861135b8a55c0b186",
  "status": "FP16 candidate configuration; not yet frozen for final experiments"
}
[qa] TRAIN_07446 | input=39 tokens | time=1.340s | answer='The original sound of music was performed by the cast of "The Sound of Mu

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
from google.colab import files
files.upload()  # run_colab_smoke_v2.py


Saving run_colab_smoke_v2.py to run_colab_smoke_v2.py


{'run_colab_smoke_v2.py': b'"""Run in the existing Colab session with: %run -i run_colab_smoke_v2.py\n\nReuse smoke_inputs.csv from round 1. Load the cached model from an exact snapshot,\nstrengthen task output instructions, and test a 256-token output allowance.\nThis diagnostic never reads references and does not train model parameters.\n"""\n\nimport csv\nimport gc\nimport hashlib\nimport json\nimport re\nimport shutil\nimport time\nfrom collections import Counter\nfrom datetime import datetime, timezone\nfrom importlib.metadata import version\nfrom pathlib import Path\n\nimport torch\nfrom huggingface_hub import scan_cache_dir\nfrom transformers import AutoModelForCausalLM, AutoTokenizer, GenerationConfig\n\n\nif not torch.cuda.is_available():\n    raise RuntimeError("A CUDA GPU is required for this diagnostic.")\n\nMODEL_ID = "Qwen/Qwen3-1.7B"\nPROMPT_VERSION = "task_output_rules_v2"\nloaded_revision = getattr(getattr(globals().get("fp16_model"), "config", None), "_commit_hash", N

In [ ]:
%run -i run_colab_smoke_v2.py

Pinned cached model revision: 70d244cc86ccca08cf5af4e1e306ecf908b1ad5e


Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

Environment: {
  "model_id": "Qwen/Qwen3-1.7B",
  "model_revision": "70d244cc86ccca08cf5af4e1e306ecf908b1ad5e",
  "revision_selection": "sole_cached_weight_snapshot",
  "explicit_snapshot_load_path": "/root/.cache/huggingface/hub/models--Qwen--Qwen3-1.7B/snapshots/70d244cc86ccca08cf5af4e1e306ecf908b1ad5e",
  "snapshot_config_sha256": {
    "config.json": "1ddb5b89ebc90dcb417a45c213d818577e65976454d29385c8f6140771d95197",
    "generation_config.json": "2325da0f15bb848e018c5ae071b7943332e9f871d6b60e2ed22ca97d4cb993d2",
    "tokenizer.json": "aeb13307a71acd8fe81861d94ad54ab689df773318809eed3cbe794b4492dae4",
    "tokenizer_config.json": "d5d09f07b48c3086c508b30d1c9114bd1189145b74e982a265350c923acd8101"
  },
  "tokenizer_id": "Qwen/Qwen3-1.7B",
  "tokenizer_revision": "70d244cc86ccca08cf5af4e1e306ecf908b1ad5e",
  "precision": "float16",
  "quantized": false,
  "gpu": "Tesla T4",
  "gpu_capacity_gib": 14.56219482421875,
  "torch": "2.11.0+cu128",
  "cuda": "12.8",
  "packages": {
    "trans

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
from google.colab import files
from pathlib import PurePosixPath
import zipfile

uploaded = files.upload()
assert set(uploaded) == {"step4_1_evaluation_demo.zip"}

with zipfile.ZipFile("step4_1_evaluation_demo.zip") as archive:
    for name in archive.namelist():
        path = PurePosixPath(name)
        assert not path.is_absolute() and ".." not in path.parts
        assert path.parts[0] == "step4_evaluation_demo"
    archive.extractall("/content")

Saving step4_1_evaluation_demo.zip to step4_1_evaluation_demo.zip


In [ ]:
%run /content/step4_evaluation_demo/project/step4_evaluation/run_evaluation_demo.py

{
  "status": "STEP4_1_EVALUATION_DEMO_PASSED",
  "count": 10,
  "bundle_manifest_verified": true,
  "gpu_required": false,
  "scope": "Ten selected validation diagnostics; not full validation or test performance",
  "per_task": {
    "qa": {
      "count": 4,
      "empty_extracted_answers": 0,
      "extraction_status_counts": {
        "text": 4
      },
      "exact_match": 25.0,
      "token_f1": 43.05555555555556,
      "strict_exact_match": 0.0
    },
    "rag": {
      "count": 3,
      "empty_extracted_answers": 0,
      "extraction_status_counts": {
        "text": 3
      },
      "exact_match": 33.333333333333336,
      "token_f1": 48.148148148148145,
      "strict_exact_match": 33.333333333333336,
      "unanswerable_count": 1,
      "unanswerable_accuracy": 0.0
    },
    "math": {
      "count": 3,
      "empty_extracted_answers": 0,
      "extraction_status_counts": {
        "complete_number": 3
      },
      "numeric_accuracy": 0.0
    }
  }
}
Results: /content/step4

In [ ]:
from google.colab import files
from io import BytesIO
from pathlib import PurePosixPath
import zipfile

uploaded_step4 = files.upload()
assert len(uploaded_step4) == 1

with zipfile.ZipFile(BytesIO(next(iter(uploaded_step4.values())))) as archive:
    for name in archive.namelist():
        path = PurePosixPath(name)
        assert not path.is_absolute() and ".." not in path.parts
        assert path.parts[0] == "step4_inference_demo"
    archive.extractall("/content")

Saving step4_2_inference_demo.zip to step4_2_inference_demo.zip


In [ ]:
%run -i /content/step4_inference_demo/project/step4_inference/run_colab_inference.py

GPU: Tesla T4
Reusing the pinned FP16 model and tokenizer.


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 10 files:   0%|          | 0/10 [00:00<?, ?it/s]

[1/10] qa TRAIN_07446 | 0.814s | answer='The original sound of music stars Julie Andrews, Richard Beymer, and Christopher Walken.'
[2/10] qa TRAIN_05848 | 0.136s | answer='grain'
[3/10] qa TRAIN_02807 | 0.138s | answer='curing'
[4/10] qa TRAIN_03597 | 0.472s | answer='March 25, 2008'
[5/10] rag TRAIN_06431 | 0.520s | answer='monetisation usually viewed as an inflationary threat'
[6/10] rag TRAIN_00268 | 0.272s | answer='Aonuma'
[7/10] rag TRAIN_09043 | 0.377s | answer='1837'
[8/10] math TRAIN_04122 | 0.180s | answer='126'
[9/10] math TRAIN_01506 | 0.137s | answer='50'
[10/10] math TRAIN_00267 | 0.219s | answer='1250'
{
  "status": "STEP4_2_INFERENCE_SMOKE_PASSED",
  "scope": "Ten selected validation diagnostics; not full validation or test accuracy",
  "test_data_read": false,
  "gpu_execution": "completed",
  "bundle_reference_verified_after_generation": true,
  "model_reused": true,
  "reference_reads_during_inference": 0,
  "count": 10,
  "task_counts": {
    "qa": 4,
    "rag": 3,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
from google.colab import files
from io import BytesIO
from pathlib import PurePosixPath
import zipfile

uploaded_preflight = files.upload()
assert len(uploaded_preflight) == 1

with zipfile.ZipFile(BytesIO(next(iter(uploaded_preflight.values())))) as archive:
    for name in archive.namelist():
        path = PurePosixPath(name)
        assert not path.is_absolute() and ".." not in path.parts
        assert path.parts[0] == "step4_validation_preflight"
    archive.extractall("/content")

Saving step4_3_validation_preflight.zip to step4_3_validation_preflight.zip


In [ ]:
%run -i /content/step4_validation_preflight/project/step4_validation/run_colab_preflight.py

Loading only the pinned tokenizer; no model weights or generation.


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

{
  "status": "STEP4_3_LENGTH_PREFLIGHT_PASSED",
  "count": 907,
  "gpu_required": false,
  "model_weights_loaded": false,
  "model_generation_performed": false,
  "reference_answers_included": false,
  "test_data_read": false,
  "ready_for_generation": true,
  "scope": "Baseline target prompts for validation only; no n-shot demonstrations",
  "task_counts": {
    "rag": 320,
    "qa": 300,
    "math": 287
  },
  "input_budget": 3840,
  "output_budget": 256,
  "total_token_budget": 4096,
  "overall": {
    "count": 907,
    "min": 61,
    "median": 117,
    "mean": 205.81146637265712,
    "p90": 452,
    "p95": 496,
    "p99": 573,
    "max": 649
  },
  "per_task": {
    "qa": {
      "count": 300,
      "min": 61,
      "median": 66.0,
      "mean": 68.52,
      "p90": 77,
      "p95": 83,
      "p99": 93,
      "max": 97,
      "over_budget_count": 0
    },
    "rag": {
      "count": 320,
      "min": 172,
      "median": 412.0,
      "mean": 414.39375,
      "p90": 519,
      "p95"

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
%pip install --upgrade torch==2.11.0+cu130 torchvision==0.26.0+cu130 torchaudio==2.11.0+cu130 --index-url https://download.pytorch.org/whl/cu130

%pip install transformers==5.20.2 accelerate==1.15.0 huggingface-hub==1.33.0 tokenizers==0.23.3 safetensors==0.8.0

Looking in indexes: https://download.pytorch.org/whl/cu130
  Using cached transformers-5.20.2-py3-none-any.whl.metadata (31 kB)
Using cached transformers-5.20.2-py3-none-any.whl (12.8 MB)
  Attempting uninstall: transformers
    Found existing installation: transformers 5.21.0
    Uninstalling transformers-5.21.0:
      Successfully uninstalled transformers-5.21.0


In [ ]:
import torch
import transformers
import huggingface_hub

print("PyTorch:", torch.__version__)
print("CUDA:", torch.version.cuda)
print("Transformers:", transformers.__version__)
print("HF Hub:", huggingface_hub.__version__)

assert torch.__version__ == "2.11.0+cu130"
assert torch.version.cuda == "13.0"
assert transformers.__version__ == "5.20.2"
assert huggingface_hub.__version__ == "1.33.0"

print("Environment version checking pass")

PyTorch: 2.11.0+cu130
CUDA: 13.0
Transformers: 5.20.2
HF Hub: 1.33.0
环境版本检查通过


In [ ]:
from google.colab import files
from io import BytesIO
from pathlib import PurePosixPath
import zipfile

uploaded_validation = files.upload()
assert len(uploaded_validation) == 1

with zipfile.ZipFile(BytesIO(next(iter(uploaded_validation.values())))) as archive:
    for name in archive.namelist():
        path = PurePosixPath(name)
        assert not path.is_absolute() and ".." not in path.parts
        assert path.parts[0] == "step4_full_validation"
    archive.extractall("/content")

Saving step4_4_full_validation_baseline_evaluation_v2.zip to step4_4_full_validation_baseline_evaluation_v2.zip


In [ ]:
%run -i /content/step4_full_validation/project/step4_baseline_validation/run_colab_full_validation.py

GPU: Tesla T4
Loading the fixed FP16 model; reuse check: model or tokenizer missing


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 10 files:   0%|          | 0/10 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

Rechecking all 907 prompts against the passed tokenizer preflight.


[transformers] The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


[1/907] rag TRAIN_00010 | 0.285s | answer='UNANSWERABLE'
[2/907] qa TRAIN_00022 | 0.171s | answer='igneous'
[3/907] qa TRAIN_00032 | 0.185s | answer='St. James'
[4/907] math TRAIN_00034 | 0.143s | answer='10'
[5/907] math TRAIN_00036 | 1.670s | answer=''
[6/907] rag TRAIN_00041 | 0.287s | answer='UNANSWERABLE'
[7/907] qa TRAIN_00062 | 0.153s | answer='Freedom Day'
[8/907] rag TRAIN_00084 | 0.522s | answer='Simone Klein'
[9/907] math TRAIN_00098 | 0.229s | answer='6'
[10/907] qa TRAIN_00099 | 0.311s | answer='George IV'
[11/907] math TRAIN_00170 | 0.280s | answer='120'
[12/907] math TRAIN_00185 | 0.185s | answer='126'
[13/907] qa TRAIN_00186 | 0.157s | answer='Drake'
[14/907] rag TRAIN_00189 | 0.491s | answer='UNANSWERABLE'
[15/907] rag TRAIN_00194 | 0.882s | answer='0.5B; Apache 2.0'
[16/907] rag TRAIN_00198 | 0.361s | answer='$777 million'
[17/907] rag TRAIN_00199 | 0.370s | answer='UNANSWERABLE'
[18/907] qa TRAIN_00209 | 0.238s | answer="'LORA'"
[19/907] math TRAIN_00210 | 1.347s | a

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
from google.colab import files
from io import BytesIO
from pathlib import PurePosixPath
import zipfile

uploaded_v2 = files.upload()
assert len(uploaded_v2) == 1

with zipfile.ZipFile(BytesIO(next(iter(uploaded_v2.values())))) as archive:
    for name in archive.namelist():
        path = PurePosixPath(name)
        assert not path.is_absolute() and ".." not in path.parts
        assert path.parts[0] == "step4_evaluation_v2"
    archive.extractall("/content")

%run -i /content/step4_evaluation_v2/project/step4_evaluation_v2/rescore.py --output-root /content --download

Saving step4_evaluation_v2_rescore.zip to step4_evaluation_v2_rescore.zip
{
  "status": "EVALUATION_V2_RESCORE_PASSED",
  "count": 907,
  "source_model_output_sha256": "d83f9ca862dcc9c32a099160ebb8bb41949827276d7db1c94ce035504711a86e",
  "source_archive_sha256": "d2e551061379c6ce40b88bc38d6ba6bf7bb2d2d0463cb73b677b1f9f537aabc4",
  "original_evaluation_version": "evaluation_v1",
  "evaluation_version": "evaluation_v2",
  "model_regenerated": false,
  "test_data_read": false,
  "changed_math_extractions": 42,
  "additional_correct_math_answers": 26,
  "math_accuracy_v1": 6.2717770034843205,
  "math_accuracy_v2": 15.331010452961673,
  "text_task_scores_unchanged": true,
  "per_task": {
    "qa": {
      "count": 300,
      "empty_extracted_answers": 1,
      "extraction_status_counts": {
        "text": 299,
        "truncated": 1
      },
      "exact_match": 13.0,
      "token_f1": 19.588889487294388,
      "strict_exact_match": 10.0
    },
    "rag": {
      "count": 320,
      "empty_

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>